# NSPPK HIV Speed Benchmark

This notebook measures how `NSPPK.fit_transform(...)` runtime scales as the number of HIV graphs increases. It reuses the cached HIV `networkx` graphs from the main benchmark notebook so the graph construction step does not dominate the measurement.


In [1]:
from pathlib import Path
import pickle
import time
import urllib.request

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from statsmodels.nonparametric.smoothers_lowess import lowess

from nsppk import NSPPK

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

DATASET_URL = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/HIV.csv'
CSV_PATH = DATA_DIR / 'HIV.csv'
GRAPH_CACHE_PATH = DATA_DIR / 'HIV_networkx_graphs.pkl'

NBIT = 14
RADIUS = 1
DISTANCE = 4
CONNECTOR = 1
BALANCE_DATASET = True
LIMIT = 8000
GRAPH_COUNT_VALUES = [250, 500, 2500, 5000]
N_REPEATS = 3
PARALLEL = True
RANDOM_STATE = 42

RDLogger.DisableLog('rdApp.*')


## Load cached HIV graphs

If the graph cache does not exist yet, this notebook rebuilds it from the HIV CSV using the same discrete node and edge labels as the main benchmark.


In [2]:
def atom_to_discrete_label(atom):
    return (
        atom.GetAtomicNum(),
        atom.GetFormalCharge(),
        int(atom.GetChiralTag()),
        atom.GetTotalNumHs(),
        int(atom.GetHybridization()),
        int(atom.GetIsAromatic()),
    )


def bond_to_discrete_label(bond):
    return (
        str(bond.GetBondType()),
        int(bond.GetStereo()),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing()),
    )


def mol_to_nx_graph(mol):
    graph = nx.Graph()

    for atom in mol.GetAtoms():
        graph.add_node(atom.GetIdx(), label=atom_to_discrete_label(atom))

    for bond in mol.GetBonds():
        graph.add_edge(
            bond.GetBeginAtomIdx(),
            bond.GetEndAtomIdx(),
            label=bond_to_discrete_label(bond),
        )

    return graph


if GRAPH_CACHE_PATH.exists():
    with open(GRAPH_CACHE_PATH, 'rb') as f:
        cache = pickle.load(f)
    graphs = cache['graphs']
    labels = np.asarray(cache['labels'])
else:
    if not CSV_PATH.exists():
        urllib.request.urlretrieve(DATASET_URL, CSV_PATH)

    df = pd.read_csv(CSV_PATH)
    graphs = []
    labels = []

    for _, row in df.iterrows():
        mol = Chem.MolFromSmiles(row['smiles'])
        if mol is None:
            continue
        graphs.append(mol_to_nx_graph(mol))
        labels.append(int(row['HIV_active']))

    labels = np.asarray(labels)
    with open(GRAPH_CACHE_PATH, 'wb') as f:
        pickle.dump({'graphs': graphs, 'labels': labels, 'smiles': None}, f)

original_dataset_size = len(graphs)
if BALANCE_DATASET:
    pos_idx = np.flatnonzero(labels == 1)
    neg_idx = np.flatnonzero(labels == 0)
    rng = np.random.default_rng(RANDOM_STATE)

    if len(pos_idx) <= len(neg_idx):
        minority_idx, majority_idx = pos_idx, neg_idx
    else:
        minority_idx, majority_idx = neg_idx, pos_idx

    n_minority = len(minority_idx)
    balanced_size = 2 * n_minority
    target_size = balanced_size if LIMIT is None else min(LIMIT, len(labels))

    selected_minority = rng.choice(minority_idx, size=n_minority, replace=False)
    selected_majority_balanced = rng.choice(majority_idx, size=n_minority, replace=False)
    selected_idx = np.concatenate([selected_minority, selected_majority_balanced])

    if target_size > balanced_size:
        remaining_majority = np.setdiff1d(majority_idx, selected_majority_balanced, assume_unique=False)
        n_extra_majority = min(target_size - balanced_size, len(remaining_majority))
        extra_majority = rng.choice(remaining_majority, size=n_extra_majority, replace=False)
        selected_idx = np.concatenate([selected_idx, extra_majority])

    selected_idx = np.sort(selected_idx)
    graphs = [graphs[i] for i in selected_idx]
    labels = labels[selected_idx]
elif LIMIT is not None:
    graphs = graphs[:LIMIT]
    labels = labels[:LIMIT]

available_graphs = len(graphs)
graph_count_values = [n for n in GRAPH_COUNT_VALUES if n <= available_graphs]
if graph_count_values[-1] != available_graphs:
    graph_count_values.append(available_graphs)

print('original cached molecules:', original_dataset_size)
print('molecules used for speed benchmark:', available_graphs)
print('graph counts:', graph_count_values)
print('positive rate:', labels.mean())


original cached molecules: 41120
molecules used for speed benchmark: 8000
graph counts: [250, 500, 2500, 5000, 8000]
positive rate: 0.180375


## Benchmark `fit_transform` runtime vs number of graphs

For each subset size, we instantiate a fresh `NSPPK` encoder and run `fit_transform` multiple times. The reported runtime is the mean over repeats.


In [ ]:
results = []

print(f"{'graphs':>8} | {'repeat':>6} | {'seconds':>8} | {'sec/graph':>9}")
print('-' * 42)

for n_graphs in graph_count_values:
    subset_graphs = graphs[:n_graphs]
    runtimes = []

    for repeat_idx in range(1, N_REPEATS + 1):
        vectorizer = NSPPK(
            radius=RADIUS,
            distance=DISTANCE,
            connector=CONNECTOR,
            nbits=NBIT,
            dense=False,
            parallel=PARALLEL,
        )

        t0 = time.perf_counter()
        X = vectorizer.fit_transform(subset_graphs)
        runtime_sec = time.perf_counter() - t0
        runtimes.append(runtime_sec)

        print(f"{n_graphs:>8d} | {repeat_idx:>6d} | {runtime_sec:>8.2f} | {runtime_sec / n_graphs:>9.4f}")

    mean_runtime_sec = float(np.mean(runtimes))
    std_runtime_sec = float(np.std(runtimes, ddof=0))
    results.append(
        {
            'n_graphs': n_graphs,
            'mean_runtime_sec': mean_runtime_sec,
            'std_runtime_sec': std_runtime_sec,
            'mean_sec_per_graph': mean_runtime_sec / n_graphs,
            'n_features': X.shape[1],
            'parallel': PARALLEL,
            'nbits': NBIT,
        }
    )

results_df = pd.DataFrame(results)
results_df


  graphs | repeat |  seconds | sec/graph
------------------------------------------
     250 |      1 |     7.65 |    0.0306
     250 |      2 |     6.26 |    0.0250
     250 |      3 |     6.85 |    0.0274
     500 |      1 |     9.57 |    0.0191
     500 |      2 |     7.69 |    0.0154
     500 |      3 |     5.63 |    0.0113
    2500 |      1 |    20.78 |    0.0083
    2500 |      2 |    26.49 |    0.0106
    2500 |      3 |    28.97 |    0.0116


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

x = results_df['n_graphs'].to_numpy()
runtime_values = results_df['mean_runtime_sec'].to_numpy()
sec_per_graph_values = results_df['mean_sec_per_graph'].to_numpy()

runtime_loess = lowess(runtime_values, x, frac=0.5, return_sorted=True)
sec_per_graph_loess = lowess(sec_per_graph_values, x, frac=0.5, return_sorted=True)

runtime_line, = axes[0].plot(x, runtime_values, marker='o', alpha=0.5, label='Mean runtime')
axes[0].fill_between(
    x,
    results_df['mean_runtime_sec'] - results_df['std_runtime_sec'],
    results_df['mean_runtime_sec'] + results_df['std_runtime_sec'],
    color=runtime_line.get_color(),
    alpha=0.15,
    label='Std. dev.',
)
axes[0].plot(runtime_loess[:, 0], runtime_loess[:, 1], color=runtime_line.get_color(), linewidth=4, alpha=1.0, label='Runtime LOESS')
axes[0].set_xlabel('Number of graphs')
axes[0].set_ylabel('Seconds')
axes[0].set_title('NSPPK fit_transform runtime')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

sec_line, = axes[1].plot(x, sec_per_graph_values, marker='o', alpha=0.5, color='tab:green', label='Mean sec/graph')
axes[1].plot(sec_per_graph_loess[:, 0], sec_per_graph_loess[:, 1], color=sec_line.get_color(), linewidth=4, alpha=1.0, label='Sec/graph LOESS')
axes[1].set_xlabel('Number of graphs')
axes[1].set_ylabel('Seconds per graph')
axes[1].set_title('NSPPK runtime per graph')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()
